In [ ]:
from pathlib import Path
import importlib.util

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from matplotlib.colors import LinearSegmentedColormap
from tokenizers import Tokenizer
from tqdm import tqdm
from transformers import AutoConfig, AutoModelForCausalLM, PreTrainedTokenizerFast

In [ ]:
TEXT = "RNA, Spodoptera litura, LOC111353161, glutaminase liver isoform, mitochondrial isoform X1<seq><gene>CATTTGATTGTTTTGTAATATATTGTATACTAGATAATATATGATGGCATAGGTTCCCGTCTATTACTAATTTGGCACTCCCAGATGTAGTATATTTATAAACGGGAGACAGAAGGAGA<CDS>ATGGCTCAGTGTGTCGTGCGCGGAAGTGTCGGGCATTTCGCACGATATTCGGCTCAGTTCTCGCAATTCACGGCGCGTACGCATCCCTCGCAGCCGTCCCGCAACTCTGGCATGCGCTACTTGTACAACAGTGGCTTCGAAGAATTTATGATGGAATGTGATAACAGTTTTGAGAAATGGCGCAATTCTGCCCCGCATTTCTATTCAGTCCAGTCCCTCGACTTGAGCAGCAAAACCCGACAGTACTCGTTCGTGCATAACATCGACCCCATATCGGTGCGCGAAGGACCGCACAAGAACGCTGACGATGTTTTATTTGACATGTTTAAGAATGAAGACACGGGTCTCCTTCCCGTTGGAAAGTTTTTGGCCGCCCTGAGGACCACTGGCATAAGGAAAAATGATAATCGTGTAAAAGAAGTCATGCATAATCTATACAAGGTGCACAAAGAGTCCAACTTTGAAGGCGGTTCACCGGAGACATTGAAACTAGACAGGAAGACCTTCAAAGAGGTGATCGCACCTAACATCGTCCTCATCACGAGGGCGTTCCGCAGTCAATTCGTAATTCCAGACTTTCAAGATTTTATCAAAGACGTGGAAGAAATGTACTGGGCTGCTAAAAGTAACACTGACGGTAAAGTGGCCAGCTACATCCCGCAGCTCGCTAGGGCCAGCTCAGAGAACTGGGGTGTCAGCATTTGTACTATAGACGGTCAACGATTTTCTATCGGTGATGTGACGGTTCCATTCACTCTACAATCGTGCAGCAAACCACTCACATACGCGATGGCACTTGAAACTCTAGGACCCGACGTGGTACACAAATATGTTGGCACGGAGCCCAGCGGCCGCAACTTCAATGAACTCGTTCTGGATTATAACATGAAGCCACATAACCCGATGATAAACGCGGGAGCAATTCTAGTATGCTCATTATTGAAGACCTTGGACCGGCCCGAGATGACTCTCGCTGAAAAGTTTGATTACGTCATGTCGTTCTTTAGTCGGCTTGCAGGCAACGAAAGCCTTGGTTTCAACAATGCAGTATTTTTATCTGAACGCGAAGCTGCCGACAGAAACTATGCACTTGGCTTCTACATGCGAGAGCATAAATGTTATCCGGAAAAAACTAATTTCCGTGAATGTATGGACTTCTATTTCCAGTGTTGCTCAATGGAAGCCAACTGCGACATCATGTCTATTATGGCTGCTACTCTGGCCAATGGTGGTATCTGTCCCATCACAGATGAAAAGGTACTACGGCCCGACTCGGTACGCAACGTGCTTTCTCTGATGCATAGCTGTGGAATGTACGACTACTCAGGACAGTTTGCATTCAAAGTGGGACTTCCTGCAAAGTCTGGCGTATCTGGTGCTATGCTCATAGTTGTCCCCAATGTTATGGGTATTTGCACATGGTCGCCACCTTTGGACCCTCTGGGAAACAGCTGCCGAGGCTTGCAATTCTGTGATGACATGATTAAGAGGTTCAACTTCCACAAGTACGATAACATTCGTTACGCCAGCCACAAGAAGGATCCTCGTCGGTACAAGTTCGAGTCTGCGGGTCTCAACATTGTCAACCTCTTGTTCTCGGCTGCTAGTGGTGATCTTAGCGCTCTACGACGCCACCACCTCTCCGGTATGGACATGACTCTGTCGGACTACGACGGTCGCACTGCGCTACATCTGGCAGCTGCAGAGGGTCACCTATCCTGCGTAGACTTCCTGCTCGCTCAGTGCGGAGTTCCACACGACCCTCGCGATCGCTGGGGCAGCCGACCACTTAACGAGGCCGAGACCTTCGGACACACCGCTGTCGTACAATATTTGAAGGAATGGGAGCAGACGCACCCTACCGACAAAACCGTGGGCGCAGCTGCAGGCGCGTCAGATGTCGATGACATATTGAAAGTTAAACCAGACGCAGATGACGCTGTTAAAGACCCCAGCATCCAGGAGATTGTATCCAGGAATGGAGACAAAGTGCAATAG</CDS>AATCTCTACTGGGATGATGTAATGAGATGATCTCGTGACTATGCGTCACAAAATAGTTTTTGAAGGTGACTCCGCATTGTGTGCAGATTATGTTTCTTTCTGTAGCAGTTATGTGAACATAAGAAAAGTTGCTAATAGAAATAAAAACGCATGATTCATAGAATGCAGAGCTGATCTTAAAGGAAAGAAAATCAAGTCCTACGTATCTATTTGTAGATACGGTTAAGGGTATGTGTTATCGCCCAAAACATATTTATTTAAGTAGAAAATATTGCATTTTACGTTAGGACAGCTTTGGCCGTGAATATTTTCAAATGTCCTCATGAAATTATACCTGGCTTTTAGAAGGTATTTTATACCGATATTTACAAAATAACTCTACAATTATGAGAATTTGTGTACATGAGTATGGTTCGAAGAAGTAAATGCTTTGATATGGAAGTTATTCAGCACTAAGCTCAAGGGTGTGATATTAAAGTAACATGGAAGCACAAGGCAACATCTGGTGAGGAGTGAACAACGGGAGAGATGCTTAGAGAAATTAACTAGTCTGCTGAGAACTTGAGAAGATTGGCTAAACAAAACTGTGTGCATTTTGTGTGAAATGTAATCTCTTTCATAGATTATTTTAAATGTGAAGATTTTATGCAATATGTATTCATAAGTAGGTAGTGTGTAGCTACTGATTTTAGAGAAATTTATTAGATAAAATACATAAATAGGCATGTACATTAACCATGGTATCTCAGTGTTAATCTCTCTACTAATCGTCTATCTTTAAGCATCTGATTGTAATAAAAATAACTCCCAAAACTGTATTTTAACATTCCATATGATATAAATGACAGGCTTGCTGTACCAGGAACAATCAAGTGTTTATTCTAGAAGAATATAAAAACTTGAAGGAATGTTAAAAATTATTAATTAAATAATAATGTTAATAAATATAAGTTATCACTTATATTCTGACAATAAATAGGAGCTATTCTATGCCATTATTGATGTTGTCCAAATGTTTGTCAATTAAAT</gene></seq><eos>"

In [ ]:
PROJECT_ROOT = Path("..")
MODEL_DIR = PROJECT_ROOT / "model" / "GenNA"
TOKENIZER_PATH = PROJECT_ROOT / "configs" / "tokenizer.json"
DATA_FILE = Path("")

TARGET_LAYER = -18
SAMPLE_SIZE = 100
MAX_LENGTH = 4096

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.bfloat16 if DEVICE.type == "cuda" else torch.float32
FLASH_ATTN_AVAILABLE = DEVICE.type == "cuda" and importlib.util.find_spec("flash_attn") is not None

if not MODEL_DIR.exists():
    raise FileNotFoundError(f"Required path not found: {MODEL_DIR}")
if not TOKENIZER_PATH.exists():
    raise FileNotFoundError(f"Required path not found: {TOKENIZER_PATH}")

In [ ]:
sns.set_theme(style="white")
plt.rcParams.update({
    "figure.dpi": 300,
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
    "font.size": 12,
    "axes.labelsize": 14,
    "axes.titlesize": 16,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.linewidth": 1.2,
    "axes.edgecolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "xtick.major.width": 1.2,
    "ytick.major.width": 1.2,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "xtick.major.size": 5,
    "ytick.major.size": 5,
    "axes.unicode_minus": False,
})

HEATMAP_COLORS = ["#FFFFFF", "#FFF3E0", "#FFB74D", "#F57C00", "#BF360C"]
HEATMAP_CMAP = LinearSegmentedColormap.from_list("attention_orange_red", HEATMAP_COLORS, N=256)
BOX_PLOT_PALETTE = {
    "Prompt": "#D6604D",
    "Tag": "#BABABA",
    "5' UTR": "#F4A582",
    "CDS": "#00897B",
    "3' UTR": "#2166AC",
}
BOX_SOURCE_ORDER = ["Prompt", "Tag", "5' UTR", "CDS", "3' UTR"]

In [ ]:
def build_tokenizer(tokenizer_path: Path) -> PreTrainedTokenizerFast:
    tokenizer_core = Tokenizer.from_file(str(tokenizer_path))
    tokenizer = PreTrainedTokenizerFast(
        tokenizer_object=tokenizer_core,
        unk_token="<unk>",
        pad_token="<pad>",
        eos_token="<eos>",
        model_max_length=MAX_LENGTH,
    )
    return tokenizer


def load_model(model_dir: Path, device: torch.device, dtype: torch.dtype):
    config = AutoConfig.from_pretrained(model_dir)
    model_kwargs = {
        "config": config,
        "torch_dtype": dtype,
        "attn_implementation": "eager",
    }
    model = AutoModelForCausalLM.from_pretrained(model_dir, **model_kwargs).to(device)
    model.eval()
    return model


def find_all_occurrences(tokens: list[str], keyword: str) -> list[int]:
    return [i for i, token in enumerate(tokens) if keyword.lower() in token.lower()]


def get_region_indices(tokens: list[str]) -> dict[str, tuple[int, int]] | None:
    genes = find_all_occurrences(tokens, "gene")
    cdss = find_all_occurrences(tokens, "cds")
    if len(genes) < 2 or len(cdss) < 2:
        return None

    idx_gene_start = genes[0]
    idx_gene_end = genes[-1]
    idx_cds_start = cdss[0]
    idx_cds_end = cdss[-1]

    if not (idx_gene_start < idx_cds_start < idx_cds_end < idx_gene_end):
        return None

    return {
        "Prompt": (0, idx_gene_start),
        "5'UTR": (idx_gene_start + 1, idx_cds_start),
        "CDS": (idx_cds_start + 1, idx_cds_end),
        "3'UTR": (idx_cds_end + 1, idx_gene_end),
        "Tag_Gene_Start": (idx_gene_start, idx_gene_start + 1),
        "Tag_CDS_Start": (idx_cds_start, idx_cds_start + 1),
        "Tag_CDS_End": (idx_cds_end, idx_cds_end + 1),
        "Tag_Gene_End": (idx_gene_end, idx_gene_end + 1),
    }


def compute_attention_outputs(model, tokenizer, text: str, target_layer: int = -18):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
    input_ids = inputs["input_ids"].to(model.device)
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
    with torch.no_grad():
        outputs = model(input_ids=input_ids, output_attentions=True)
    attention = outputs.attentions[target_layer]
    avg_attention = attention[0].mean(dim=0)
    return tokens, avg_attention, attention


def calc_full_breakdown(attn_matrix: torch.Tensor, regions: dict[str, tuple[int, int]], target_region_name: str):
    tgt_start, tgt_end = regions[target_region_name]
    if tgt_end <= tgt_start:
        return None

    sub_attn = attn_matrix[tgt_start:tgt_end, :]
    stats = {}
    total_content_weight = 0.0

    for region_name in ["Prompt", "5'UTR", "CDS", "3'UTR"]:
        start, end = regions[region_name]
        if end > start:
            weight = sub_attn[:, start:end].sum(dim=1).mean().item()
        else:
            weight = 0.0
        stats[region_name] = weight
        total_content_weight += weight

    stats["StructTag"] = max(0.0, 1.0 - total_content_weight)
    return stats


def display_name_map(name: str) -> str:
    mapping = {
        "Prompt": "Prompt",
        "StructTag": "Tag",
        "5'UTR": "5' UTR",
        "CDS": "CDS",
        "3'UTR": "3' UTR",
    }
    return mapping.get(name, name)

In [ ]:
tokenizer = build_tokenizer(TOKENIZER_PATH)
model = load_model(MODEL_DIR, DEVICE, DTYPE)
print(f"Model loaded on {DEVICE}")

In [ ]:
if TEXT:
    tokens, avg_attention, raw_attention = compute_attention_outputs(model, tokenizer, TEXT, TARGET_LAYER)
    regions = get_region_indices(tokens)
else:
    tokens, avg_attention, raw_attention, regions = None, None, None, None

print(f"Single-sequence analysis ready: {tokens is not None}")

In [ ]:
def plot_attention_heatmap(attention: torch.Tensor, tokens: list[str], regions: dict[str, tuple[int, int]] | None) -> None:
    if attention is None or tokens is None:
        print("No single-sequence input available.")
        return

    avg_attn = attention[0].mean(dim=0).float().cpu().numpy()
    seq_len = len(tokens)

    if regions is None:
        print("Region tags were not found. Plotting heatmap without region overlays.")
        fig, ax = plt.subplots(figsize=(8, 8), dpi=150)
        heatmap = sns.heatmap(
            avg_attn,
            cmap=HEATMAP_CMAP,
            ax=ax,
            cbar=True,
            rasterized=True,
            vmin=0,
            vmax=0.005,
            xticklabels=False,
            yticklabels=False,
            square=True,
            cbar_kws={"shrink": 0.7, "aspect": 20, "pad": 0.03},
        )
        cbar = heatmap.collections[0].colorbar
        cbar.outline.set_linewidth(0.8)
        cbar.ax.tick_params(labelsize=12, width=0.6, length=3)
        cbar.set_label("Attention Weight", fontsize=16, labelpad=10)
        ax.set_xlabel("Token Index (Key)", fontsize=16, labelpad=10)
        ax.set_ylabel("Token Index (Query)", fontsize=16, labelpad=10)
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(1.0)
            spine.set_color("black")
        ax.tick_params(axis="both", which="both", length=0)
        plt.tight_layout()
        plt.show()
        return

    idx_gene_start = regions["Tag_Gene_Start"][0]
    idx_cds_start = regions["Tag_CDS_Start"][0]
    idx_cds_end = regions["Tag_CDS_End"][0]
    idx_gene_end = regions["Tag_Gene_End"][0]

    overlay_regions = [
        ("5' UTR", idx_gene_start, idx_cds_start, "#404040", "-", 2.0, 30),
        ("CDS", idx_cds_start, idx_cds_end, "#00897B", "-", 2.5, 0),
        ("3' UTR", idx_cds_end, idx_gene_end, "#404040", "-", 2.0, 0),
    ]

    fig, ax = plt.subplots(figsize=(8, 8), dpi=150)
    heatmap = sns.heatmap(
        avg_attn,
        cmap=HEATMAP_CMAP,
        ax=ax,
        cbar=True,
        rasterized=True,
        vmin=0,
        vmax=0.005,
        xticklabels=False,
        yticklabels=False,
        square=True,
        cbar_kws={"shrink": 0.7, "aspect": 20, "pad": 0.03},
    )

    cbar = heatmap.collections[0].colorbar
    cbar.outline.set_linewidth(0.8)
    cbar.ax.tick_params(labelsize=12, width=0.6, length=3)
    cbar.set_label("Attention Weight", fontsize=16, labelpad=10)

    prompt_limit = idx_gene_start
    if prompt_limit > 0:
        rect = patches.Rectangle((0, 0), prompt_limit, seq_len, linewidth=2, edgecolor="#1565C0", facecolor="none", linestyle="-")
        ax.add_patch(rect)
        ax.text(
            prompt_limit / 2,
            min(100, seq_len / 2),
            "Prompt",
            color="#1565C0",
            ha="center",
            va="center",
            fontsize=14,
            fontweight="bold",
            rotation=90,
            bbox=dict(boxstyle="square,pad=0.36", facecolor="white", alpha=0.95, edgecolor="none"),
            zorder=100,
        )

    for label, start, end, color, style, lw, x_offset in overlay_regions:
        width = end - start
        if width <= 0:
            continue
        rect = patches.Rectangle((start, start), width, width, linewidth=lw, edgecolor=color, facecolor="none", linestyle=style)
        ax.add_patch(rect)
        text_x = start + width / 2 + x_offset
        ax.text(text_x, max(0, start - 15), label, color=color, ha="center", va="bottom", fontsize=14, fontweight="bold", zorder=100)

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.0)
        spine.set_color("black")

    ax.set_xlabel("Token Index (Key)", fontsize=16, labelpad=10)
    ax.set_ylabel("Token Index (Query)", fontsize=16, labelpad=10)
    ax.tick_params(axis="both", which="both", length=0)
    plt.tight_layout()
    plt.show()


plot_attention_heatmap(raw_attention, tokens, regions)